# Does Balance Change How You Talk?

**Thursday Masking Day · Tilburg Multiscale Summerschool 2026 · Take-home notebook**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WimPouw/TilburgMultiscaleSummerschool2026/blob/main/thursday-masking/notebooks/08_balance_language.ipynb)

---

In the BalanceCorpus, one participant stands on a balance board while describing words —  
the other stands on solid ground. Does the physical load change *how* people talk?

Embodied cognition research suggests it might: managing balance consumes cognitive  
resources, potentially pushing speech toward simpler, more concrete words, more  
hesitations, shorter sentences.

This notebook uses **automatic speech recognition** (Whisper) to transcribe  
BalanceCorpus audio clips, then compares language across conditions using  
basic corpus analysis techniques.

### What you will do

| Step | Task |
|------|------|
| 1 | Transcribe audio clips with Whisper |
| 2 | Score words for concreteness using published norms |
| 3 | Count hedges, fillers, and repairs |
| 4 | Compare POS distributions across conditions |
| 5 | Test whether difficulty predicts language measures |

## Setup

In [ ]:
import sys, warnings, re, collections
warnings.filterwarnings('ignore')

!{sys.executable} -m pip install -q openai-whisper spacy pandas matplotlib
!{sys.executable} -m spacy download en_core_web_sm -q

import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#161616', 'axes.facecolor': '#262626',
    'axes.edgecolor': '#525252', 'axes.labelcolor': '#c6c6c6',
    'xtick.color': '#6f6f6f', 'ytick.color': '#6f6f6f',
    'text.color': '#f4f4f4', 'grid.color': '#393939', 'grid.linestyle': '--',
})

nlp = spacy.load('en_core_web_sm')
print('Ready.')

## 1 — Transcribe audio clips with Whisper

OpenAI Whisper is a free, local speech-to-text model. The `tiny` model runs on CPU  
in seconds. Point `AUDIO_FILES` at any `.wav` files from the BalanceCorpus,  
or use the synthetic fallback transcripts below.

In [ ]:
# Set these to actual paths if you have the BalanceCorpus audio files
AUDIO_FILES = {
    # 'board_doughnut': Path('../data/raw/.../103_203_12_..._doughnut_board.wav'),
    # 'ground_spinach':  Path('../data/raw/.../103_203_33_..._spinach_ground.wav'),
}

# Synthetic fallback transcripts (realistic approximations)
FALLBACK_TRANSCRIPTS = {
    'board_doughnut': (
        "It's um, it's like a ring shape, kind of, you know, sweet, "
        "fried dough thing, people eat it for breakfast, it has a hole, "
        "wait no, anyway it's sweet and fried, glazed sometimes"
    ),
    'ground_doughnut': (
        "It's a fried ring of dough that's usually glazed or covered in "
        "sugar. You find them at bakeries. They have a distinctive circular "
        "shape with a hole in the middle."
    ),
    'board_tiger': (
        "Um, big cat, like orange, has stripes, kind of, lives in jungle, "
        "very dangerous, um, predator, Asia mainly"
    ),
    'ground_tiger': (
        "It's a large wild cat with orange and black stripes. "
        "An apex predator found in Asian jungles and forests."
    ),
    'board_hunger': (
        "Um, it's like when you need food, kind of a feeling, "
        "stomach thing, maybe growling, sort of when you haven't eaten"
    ),
    'ground_hunger': (
        "The physiological need for food. A sensation that motivates "
        "eating behaviour. Also used metaphorically for strong desire."
    ),
}

transcripts = {}

if AUDIO_FILES:
    import whisper
    model = whisper.load_model('tiny')
    for label, path in AUDIO_FILES.items():
        result = model.transcribe(str(path), language='en')
        transcripts[label] = result['text'].strip()
        print(f'{label}: {transcripts[label][:80]}...')
else:
    transcripts = FALLBACK_TRANSCRIPTS
    print('Using synthetic fallback transcripts.')
    print('To use real audio, set AUDIO_FILES above and re-run.')
    print()
    for label, text in transcripts.items():
        print(f'{label}: {text[:80]}...')

## 2 — Basic text statistics per clip

Word count, unique words (type-token ratio), and average word length —  
the simplest measures that don't require any external resources.

In [ ]:
def basic_stats(text: str) -> dict:
    tokens = [t.lower() for t in re.findall(r"[a-z']+", text)]
    words  = [t for t in tokens if t not in {'um','uh','like','kind','sort','you','know'}]
    return {
        'tokens':    len(tokens),
        'types':     len(set(tokens)),
        'ttr':       round(len(set(tokens)) / max(len(tokens), 1), 3),
        'avg_len':   round(np.mean([len(w) for w in words]) if words else 0, 2),
    }

stats = pd.DataFrame(
    [{**{'clip': label, 'condition': label.split('_')[0]},
      **basic_stats(text)}
     for label, text in transcripts.items()]
)
print(stats.to_string(index=False))

## 3 — Hedge and filler counting

Hedges ("kind of", "sort of", "maybe") and fillers ("um", "uh") are markers  
of uncertainty or processing difficulty. Do they appear more in the board condition?

In [ ]:
FILLERS = {'um', 'uh', 'er', 'ah', 'hmm'}
HEDGES  = {'kind of', 'sort of', 'like a', 'kind of a', 'maybe', 'probably',
            'i think', 'you know', 'something like', 'a bit', 'sort of like'}

def count_disfluency(text: str) -> dict:
    lower = text.lower()
    tokens = re.findall(r"[a-z']+", lower)
    filler_n = sum(1 for t in tokens if t in FILLERS)
    hedge_n  = sum(1 for h in HEDGES if h in lower)
    return {
        'fillers':         filler_n,
        'hedges':          hedge_n,
        'disfluency_rate': round((filler_n + hedge_n) / max(len(tokens), 1), 3),
    }

dis = pd.DataFrame(
    [{**{'clip': label, 'condition': label.split('_')[0]},
      **count_disfluency(text)}
     for label, text in transcripts.items()]
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ['fillers', 'hedges', 'disfluency_rate']):
    colors = ['#da1e28' if c == 'board' else '#42be65'
              for c in dis['condition']]
    ax.bar(dis['clip'], dis[col], color=colors)
    ax.set_title(col, color='#f4f4f4')
    ax.tick_params(axis='x', rotation=30, labelsize=7)

legend = [
    mpatches.Patch(color='#da1e28', label='board (balance)'),
    mpatches.Patch(color='#42be65', label='ground (stable)'),
]
axes[2].legend(handles=legend, framealpha=0.2, facecolor='#262626',
               labelcolor='#f4f4f4', fontsize=8)
plt.suptitle('Disfluency markers by condition', color='#f4f4f4', fontsize=12)
plt.tight_layout()
plt.show()
print(dis.to_string(index=False))

## 4 — Part-of-speech distribution

Are clue-givers using more nouns (object properties), verbs (functions),  
or adjectives (attributes)?  Does the ratio shift on the balance board?

In [ ]:
POS_KEEP = {'NOUN', 'VERB', 'ADJ', 'ADV', 'INTJ'}

def pos_profile(text: str) -> dict:
    doc = nlp(text)
    counts = collections.Counter(t.pos_ for t in doc if t.pos_ in POS_KEEP)
    total  = sum(counts.values()) or 1
    return {pos: round(counts[pos] / total, 3) for pos in POS_KEEP}

pos_data = pd.DataFrame(
    [{**{'clip': label, 'condition': label.split('_')[0]},
      **pos_profile(text)}
     for label, text in transcripts.items()]
)

fig, ax = plt.subplots(figsize=(10, 5))
pos_cols = list(POS_KEEP)
x = np.arange(len(pos_data))
width = 0.14
pos_colors = {'NOUN': '#78a9ff', 'VERB': '#42be65', 'ADJ': '#f1c21b',
              'ADV': '#ff832b', 'INTJ': '#da1e28'}

for i, pos in enumerate(pos_cols):
    ax.bar(x + i * width, pos_data[pos], width, label=pos,
           color=pos_colors[pos], alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(pos_data['clip'], rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Proportion of content words')
ax.set_title('Part-of-speech profile per clip', color='#f4f4f4', fontsize=12)
ax.legend(facecolor='#262626', labelcolor='#f4f4f4', framealpha=0.3, fontsize=9)
plt.tight_layout()
plt.show()

## 5 — Concreteness: do people use more concrete words under load?

Brysbaert et al. (2014) collected concreteness ratings for ~40k English words  
(1 = abstract, 5 = concrete). We look up each content word and average the scores.

Hypothesis: balance board → cognitive load → shift toward more concrete, simpler words.

In [ ]:
# Download Brysbaert norms (public domain, hosted on OSF)
NORMS_URL = ('https://raw.githubusercontent.com/WimPouw/'
             'TilburgMultiscaleSummerschool2026/main/'
             'Datasets/BalanceCorpus/metadata.csv')  # placeholder — see note below

# Concreteness norms are not in the repo; we use a built-in sample
# Source: Brysbaert, Warriner & Kuperman (2014), Behavior Research Methods
CONCRETENESS = {
    'ring': 4.61, 'sweet': 3.91, 'fried': 4.29, 'dough': 4.52,
    'breakfast': 3.67, 'glazed': 3.84, 'sugar': 4.81, 'cake': 4.91,
    'cat': 4.94, 'orange': 4.57, 'stripe': 4.06, 'jungle': 4.57,
    'predator': 3.69, 'feline': 3.85, 'wild': 3.19, 'claw': 4.83,
    'food': 3.96, 'stomach': 4.92, 'feel': 2.39, 'need': 2.13,
    'feeling': 2.22, 'sensation': 2.67, 'eat': 3.38, 'physiological': 1.64,
    'large': 2.64, 'animal': 4.39, 'forest': 4.52, 'asian': 3.17,
    'growl': 3.92, 'desire': 2.04, 'behaviour': 2.18, 'metaphor': 1.59,
    'bake': 3.82, 'circular': 3.12, 'shape': 3.38, 'distinctive': 1.88,
    'bakery': 4.54, 'apex': 2.29, 'dangerous': 2.35, 'fat': 3.93,
}

def mean_concreteness(text: str, norms: dict) -> float:
    doc = nlp(text.lower())
    scores = [
        norms[t.lemma_]
        for t in doc
        if t.pos_ in {'NOUN','VERB','ADJ'} and t.lemma_ in norms
    ]
    return round(np.mean(scores), 3) if scores else float('nan')

conc = pd.DataFrame([
    {'clip': label,
     'condition': label.split('_')[0],
     'concreteness': mean_concreteness(text, CONCRETENESS)}
    for label, text in transcripts.items()
])

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#da1e28' if c == 'board' else '#42be65' for c in conc['condition']]
ax.bar(conc['clip'], conc['concreteness'], color=colors)
ax.axhline(conc['concreteness'].mean(), color='#8d8d8d', linestyle='--', label='mean')
ax.set_ylabel('Mean concreteness (Brysbaert scale 1–5)')
ax.set_title('Word concreteness per clip', color='#f4f4f4', fontsize=12)
legend = [
    mpatches.Patch(color='#da1e28', label='board'), mpatches.Patch(color='#42be65', label='ground'),
]
ax.legend(handles=legend, framealpha=0.2, facecolor='#262626', labelcolor='#f4f4f4')
ax.tick_params(axis='x', rotation=20, labelsize=8)
plt.tight_layout()
plt.show()
print(conc.to_string(index=False))

## 6 — Lexical diversity across clips

Type-token ratio (TTR) measures how many *different* words a speaker uses relative  
to their total words. A lower TTR means more repetition — potentially a sign of  
reduced vocabulary access under cognitive load.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#da1e28' if c == 'board' else '#42be65' for c in stats['condition']]
ax.bar(stats['clip'], stats['ttr'], color=colors)
ax.set_ylabel('Type-Token Ratio')
ax.set_title('Lexical diversity (TTR) per clip', color='#f4f4f4', fontsize=12)
legend = [
    mpatches.Patch(color='#da1e28', label='board'),
    mpatches.Patch(color='#42be65', label='ground'),
]
ax.legend(handles=legend, framealpha=0.2, facecolor='#262626', labelcolor='#f4f4f4')
ax.tick_params(axis='x', rotation=20, labelsize=8)
plt.tight_layout()
plt.show()

print('\nNote: TTR is sensitive to text length — longer texts naturally have lower TTR.')
print('For a fair comparison, use clips of similar length.')

---
## What just happened?

| Concept | Where it appeared |
|---------|------------------|
| ASR / speech-to-text | Whisper transcription |
| Tokenisation | `re.findall` + spaCy |
| Type-token ratio | Basic corpus metric |
| Disfluency analysis | Filler + hedge counting |
| Part-of-speech tagging | spaCy `.pos_` |
| Psycholinguistic norms | Brysbaert concreteness ratings |

---
## Go further

- **Transcribe more clips** — Whisper on all board/ground pairs for one target word;  
  test whether board clips consistently differ from ground clips
- **Download the full Brysbaert norms** from [osf.io/jhzk5](https://osf.io/jhzk5)  
  and replace the hand-typed sample above
- **Sentence length** — split each transcript at punctuation and count words per sentence
- **Valence/arousal** — use the ANEW or NRC lexicon to score emotional tone per clip
- **Connect to gestures** — do clips with more hedges also have more gesture (from notebook 07)?

### Connection to Thursday

- **Notebook 07** — the taboo geometry gives you a "semantic difficulty" score per target;  
  correlate it with the language measures from this notebook
- **Notebook 09** — the transcripts produced here feed directly into the de-identification pipeline